# Chapter 1 Computational Lab
## Combinatorial Foundations for Probability

This interactive Colab notebook accompanies Chapter 1 of *Probability Theory with Python and AI*. It is designed as a computational laboratory rather than a list of isolated formulas.

### Learning goals

By the end of the notebook, you should be able to:

1. distinguish ordered from unordered sampling;
2. decide whether replacement is allowed;
3. compute permutations, combinations and multinomial coefficients;
4. use stars and bars and inclusion--exclusion;
5. calculate finite uniform, hypergeometric and birthday-collision probabilities;
6. test your understanding through generated exercises and a final quiz.

Run the cells from top to bottom. Change the sliders and menus, predict the answer first, and then compare your reasoning with the MathJax output.

## 0. Notebook setup

The first cell imports the standard Python tools used throughout the lab. It also defines a display helper so that numerical conclusions appear together with properly typeset mathematical formulas. No external data or nonstandard package installation is required in Google Colab.

In [ ]:
import math
import random
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass

plt.style.use("seaborn-v0_8-whitegrid")


def show_math_result(title, *latex_lines, note=None):
    """Display a titled result followed by MathJax equations."""
    display(HTML(
        f"<div style='border-left:5px solid #222;padding:8px 12px;"
        f"background:#f3f3f3;margin:8px 0'><b>{title}</b></div>"
    ))
    for line in latex_lines:
        display(Math(line))
    if note:
        display(Markdown(note))


def falling_factorial(n, k):
    if not (0 <= k <= n):
        return 0
    return math.factorial(n) // math.factorial(n - k)


def multinomial(counts):
    total = sum(counts)
    value = math.factorial(total)
    for count in counts:
        value //= math.factorial(count)
    return value


def derangement(n):
    if n == 0:
        return 1
    if n == 1:
        return 0
    d0, d1 = 1, 0
    for k in range(2, n + 1):
        d0, d1 = d1, (k - 1) * (d1 + d0)
    return d1


def onto_count(n, m):
    return sum(
        (-1) ** j * math.comb(m, j) * (m - j) ** n
        for j in range(m + 1)
    )


def birthday_collision_probability(k, n=365):
    if k > n:
        return 1.0
    no_collision = 1.0
    for j in range(k):
        no_collision *= (n - j) / n
    return 1.0 - no_collision


display(HTML(
    "<div style='padding:10px;border:1px solid #888;background:white'>"
    "<b>Setup complete.</b> The interactive controls below are ready.</div>"
))

## 1. Factorials and ordered selections

The number of linear orderings of $$n$$ distinct objects is

$$
n!=n(n-1)\cdots 2\cdot1.
$$

If only $$k$$ positions are filled without replacement, the relevant count is the falling factorial

$$
(n)_k=n(n-1)\cdots(n-k+1)=\frac{n!}{(n-k)!}.
$$

With replacement, every position again has $$n$$ choices, so the count becomes $$n^k$$. Use the controls to compare these cases.

In [ ]:
n_ordered = widgets.IntSlider(value=8, min=1, max=20, step=1, description="n")
k_ordered = widgets.IntSlider(value=3, min=0, max=20, step=1, description="k")
replacement_ordered = widgets.Checkbox(value=False, description="With replacement")
ordered_output = widgets.Output()


def update_ordered(*_):
    with ordered_output:
        clear_output(wait=True)
        n, k = n_ordered.value, k_ordered.value
        if replacement_ordered.value:
            value = n ** k
            show_math_result(
                "Ordered sampling with replacement",
                rf"n^k={n}^{{{k}}}={value:,}",
                note="Every position may contain any of the available objects.",
            )
        elif k > n:
            show_math_result(
                "Ordered sampling without replacement",
                rf"k={k}>n={n}\quad\Longrightarrow\quad (n)_k=0",
                note="There are not enough distinct objects to fill all positions.",
            )
        else:
            value = falling_factorial(n, k)
            show_math_result(
                "Ordered sampling without replacement",
                rf"(n)_k=\frac{{n!}}{{(n-k)!}}"
                rf"=\frac{{{n}!}}{{({n}-{k})!}}={value:,}",
            )


for control in (n_ordered, k_ordered, replacement_ordered):
    control.observe(update_ordered, names="value")

display(widgets.VBox([
    widgets.HBox([n_ordered, k_ordered]),
    replacement_ordered,
    ordered_output,
]))
update_ordered()

## 2. The four standard sampling schemes

Two questions determine the appropriate formula: Is order recorded? Is replacement allowed? The four answers are

$$
\begin{array}{c|c|c}
&\text{with replacement}&\text{without replacement}\\ \hline
\text{ordered}&n^k&(n)_k\\
\text{unordered}&\displaystyle\binom{n+k-1}{k}&\displaystyle\binom nk
\end{array}
$$

Choose a sampling description below. The notebook identifies the formula and evaluates it.

In [ ]:
scheme_order = widgets.Dropdown(
    options=[("Ordered", True), ("Unordered", False)],
    value=True,
    description="Order",
)
scheme_replacement = widgets.Dropdown(
    options=[("With replacement", True), ("Without replacement", False)],
    value=True,
    description="Reuse",
)
scheme_n = widgets.IntSlider(value=5, min=1, max=30, description="n")
scheme_k = widgets.IntSlider(value=3, min=0, max=30, description="k")
scheme_output = widgets.Output()


def update_scheme(*_):
    with scheme_output:
        clear_output(wait=True)
        ordered = scheme_order.value
        replace = scheme_replacement.value
        n, k = scheme_n.value, scheme_k.value

        if ordered and replace:
            value, formula = n ** k, rf"n^k={n}^{k}"
            label = "ordered, with replacement"
        elif ordered and not replace:
            value = falling_factorial(n, k)
            formula = rf"(n)_k=\frac{{n!}}{{(n-k)!}}"
            label = "ordered, without replacement"
        elif not ordered and replace:
            value = math.comb(n + k - 1, k)
            formula = rf"\binom{{n+k-1}}{{k}}=\binom{{{n+k-1}}}{{{k}}}"
            label = "unordered, with replacement"
        else:
            value = math.comb(n, k) if k <= n else 0
            formula = rf"\binom{{n}}{{k}}=\binom{{{n}}}{{{k}}}"
            label = "unordered, without replacement"

        show_math_result(
            f"Sampling scheme: {label}",
            rf"{formula}={value:,}",
            note="A count of zero means that the requested sample is impossible.",
        )


for control in (scheme_order, scheme_replacement, scheme_n, scheme_k):
    control.observe(update_scheme, names="value")

display(widgets.VBox([
    widgets.HBox([scheme_order, scheme_replacement]),
    widgets.HBox([scheme_n, scheme_k]),
    scheme_output,
]))
update_scheme()

## 3. Binomial coefficients and Pascal's identity

An $$n$$-element set has

$$
\binom nk=\frac{n!}{k!(n-k)!}
$$

subsets of size $$k$$. Separating the subsets according to whether they contain one fixed element gives Pascal's identity:

$$
\binom{n+1}{k}=\binom nk+\binom n{k-1}.
$$

The plot displays an entire row of Pascal's triangle, while the equation checks one chosen entry.

In [ ]:
pascal_n = widgets.IntSlider(value=8, min=1, max=25, description="n")
pascal_k = widgets.IntSlider(value=3, min=1, max=25, description="k")
pascal_output = widgets.Output()


def update_pascal(*_):
    with pascal_output:
        clear_output(wait=True)
        n, k = pascal_n.value, pascal_k.value
        row = [math.comb(n, j) for j in range(n + 1)]

        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.bar(range(n + 1), row, color="white", edgecolor="black", linewidth=1.3)
        ax.set_title(rf"Binomial coefficients in row $n={n}$")
        ax.set_xlabel(r"$j$")
        ax.set_ylabel(r"$\binom{n}{j}$")
        ax.set_xticks(range(n + 1))
        plt.show()

        display(Math(rf"\sum_{{j=0}}^{{{n}}}\binom{{{n}}}{{j}}=2^{{{n}}}={2**n:,}"))
        if k <= n + 1:
            left = math.comb(n + 1, k)
            right1 = math.comb(n, k) if k <= n else 0
            right2 = math.comb(n, k - 1)
            display(Math(
                rf"\binom{{{n+1}}}{{{k}}}={left:,}"
                rf"=\binom{{{n}}}{{{k}}}+\binom{{{n}}}{{{k-1}}}"
                rf"={right1:,}+{right2:,}"
            ))
        else:
            display(HTML("<b>Choose k no larger than n+1 to test Pascal's identity.</b>"))


for control in (pascal_n, pascal_k):
    control.observe(update_pascal, names="value")

display(widgets.VBox([widgets.HBox([pascal_n, pascal_k]), pascal_output]))
update_pascal()

## 4. Multinomial counting

If $$n$$ distinct positions are divided among $$m$$ labelled types with counts $$n_1,\ldots,n_m$$ and

$$
n_1+\cdots+n_m=n,
$$

then the number of compatible type sequences is

$$
\binom{n}{n_1,\ldots,n_m}=\frac{n!}{n_1!\cdots n_m!}.
$$

Enter non-negative category counts separated by commas. The output also compares the fixed-count sequences with the total number $$m^n$$ of unrestricted sequences.

In [ ]:
multinomial_input = widgets.Text(value="5, 4, 3", description="Counts")
multinomial_button = widgets.Button(description="Compute", button_style="primary")
multinomial_output = widgets.Output()


def compute_multinomial(_=None):
    with multinomial_output:
        clear_output(wait=True)
        try:
            counts = [int(item.strip()) for item in multinomial_input.value.split(",")]
            if not counts or any(value < 0 for value in counts):
                raise ValueError
        except ValueError:
            display(HTML("<b style='color:#a00'>Enter non-negative integers separated by commas.</b>"))
            return

        n, m = sum(counts), len(counts)
        value = multinomial(counts)
        lower = ",".join(str(x) for x in counts)
        denominator = "".join(rf"{x}!" for x in counts)
        show_math_result(
            "Multinomial coefficient",
            rf"\binom{{{n}}}{{{lower}}}=\frac{{{n}!}}{{{denominator}}}={value:,}",
            rf"\text{{All unrestricted sequences: }}\quad {m}^{{{n}}}={m**n:,}",
        )


multinomial_button.on_click(compute_multinomial)
display(widgets.VBox([
    widgets.HBox([multinomial_input, multinomial_button]),
    multinomial_output,
]))
compute_multinomial()

## 5. Stars and bars

The number of non-negative integer solutions of

$$
x_1+\cdots+x_m=r
$$

is

$$
\binom{r+m-1}{m-1}.
$$

If every variable must be positive, place one unit in each box first. For $$r\ge m$$, the count becomes

$$
\binom{r-1}{m-1}.
$$

For small cases, the notebook also lists every solution so that the bijection behind the formula is visible.

In [ ]:
stars_r = widgets.IntSlider(value=7, min=0, max=25, description="r")
stars_m = widgets.IntSlider(value=3, min=1, max=7, description="m")
stars_positive = widgets.Checkbox(value=False, description="Require positive values")
stars_output = widgets.Output()


def compositions(total, parts, minimum=0):
    remaining = total - parts * minimum
    if remaining < 0:
        return []

    result = []

    def build(prefix, left, slots):
        if slots == 1:
            result.append(tuple(prefix + [left + minimum]))
            return
        for value in range(left + 1):
            build(prefix + [value + minimum], left - value, slots - 1)

    build([], remaining, parts)
    return result


def update_stars(*_):
    with stars_output:
        clear_output(wait=True)
        r, m = stars_r.value, stars_m.value
        minimum = 1 if stars_positive.value else 0

        if minimum == 0:
            count = math.comb(r + m - 1, m - 1)
            formula = rf"\binom{{r+m-1}}{{m-1}}=\binom{{{r+m-1}}}{{{m-1}}}={count:,}"
            title = "Non-negative integer solutions"
        elif r >= m:
            count = math.comb(r - 1, m - 1)
            formula = rf"\binom{{r-1}}{{m-1}}=\binom{{{r-1}}}{{{m-1}}}={count:,}"
            title = "Positive integer solutions"
        else:
            count = 0
            formula = rf"r={r}<m={m}\quad\Longrightarrow\quad 0\text{{ solutions}}"
            title = "Positive integer solutions"

        show_math_result(title, formula)
        if count <= 40:
            solutions = compositions(r, m, minimum)
            display(Markdown("**All solutions:** " + ", ".join(map(str, solutions))))
        else:
            display(Markdown(f"The list contains {count:,} solutions and is not printed in full."))


for control in (stars_r, stars_m, stars_positive):
    control.observe(update_stars, names="value")

display(widgets.VBox([
    widgets.HBox([stars_r, stars_m]),
    stars_positive,
    stars_output,
]))
update_stars()

## 6. Inclusion--exclusion for three sets

For three finite sets, inclusion--exclusion states that

$$
|A\cup B\cup C|
=|A|+|B|+|C|
-|A\cap B|-|A\cap C|-|B\cap C|
+|A\cap B\cap C|.
$$

If the universe contains $$U$$ objects, the number satisfying none of the three conditions is

$$
U-|A\cup B\cup C|.
$$

Enter set and intersection sizes. The notebook warns when the supplied counts violate basic consistency requirements.

In [ ]:
ie_controls = {
    "U": widgets.IntSlider(value=100, min=1, max=200, description="|U|"),
    "A": widgets.IntSlider(value=35, min=0, max=100, description="|A|"),
    "B": widgets.IntSlider(value=28, min=0, max=100, description="|B|"),
    "C": widgets.IntSlider(value=20, min=0, max=100, description="|C|"),
    "AB": widgets.IntSlider(value=9, min=0, max=100, description="|A∩B|"),
    "AC": widgets.IntSlider(value=6, min=0, max=100, description="|A∩C|"),
    "BC": widgets.IntSlider(value=5, min=0, max=100, description="|B∩C|"),
    "ABC": widgets.IntSlider(value=2, min=0, max=100, description="|A∩B∩C|"),
}
ie_output = widgets.Output()


def update_inclusion_exclusion(*_):
    with ie_output:
        clear_output(wait=True)
        v = {name: control.value for name, control in ie_controls.items()}
        union = v["A"] + v["B"] + v["C"] - v["AB"] - v["AC"] - v["BC"] + v["ABC"]
        neither = v["U"] - union

        issues = []
        if any(v[pair] > min(v[pair[0]], v[pair[1]]) for pair in ("AB", "AC", "BC")):
            issues.append("A pairwise intersection cannot exceed either participating set.")
        if v["ABC"] > min(v["AB"], v["AC"], v["BC"]):
            issues.append("The triple intersection cannot exceed a pairwise intersection.")
        if not (0 <= union <= v["U"]):
            issues.append("The computed union must lie between zero and the universe size.")

        show_math_result(
            "Inclusion--exclusion result",
            rf"|A\cup B\cup C|={v['A']}+{v['B']}+{v['C']}"
            rf"-{v['AB']}-{v['AC']}-{v['BC']}+{v['ABC']}={union}",
            rf"|U\setminus(A\cup B\cup C)|={v['U']}-{union}={neither}",
        )
        if issues:
            display(HTML("<div style='color:#a00'><b>Consistency warning:</b><ul>" +
                         "".join(f"<li>{item}</li>" for item in issues) + "</ul></div>"))


for control in ie_controls.values():
    control.observe(update_inclusion_exclusion, names="value")

display(widgets.VBox([
    widgets.HBox([ie_controls["U"], ie_controls["A"], ie_controls["B"], ie_controls["C"]]),
    widgets.HBox([ie_controls["AB"], ie_controls["AC"], ie_controls["BC"], ie_controls["ABC"]]),
    ie_output,
]))
update_inclusion_exclusion()

## 7. Derangements and onto functions

A derangement is a permutation with no fixed point. Inclusion--exclusion gives

$$
D_n=n!\sum_{j=0}^{n}\frac{(-1)^j}{j!}.
$$

The number of onto functions from an $$n$$-element domain to an $$m$$-element target is

$$
\sum_{j=0}^{m}(-1)^j\binom mj(m-j)^n.
$$

The two panels below evaluate these formulas independently.

In [ ]:
der_n = widgets.IntSlider(value=6, min=0, max=15, description="n")
onto_n = widgets.IntSlider(value=6, min=0, max=15, description="domain n")
onto_m = widgets.IntSlider(value=3, min=1, max=10, description="target m")
der_output = widgets.Output()
onto_output = widgets.Output()


def update_derangement(*_):
    with der_output:
        clear_output(wait=True)
        n = der_n.value
        value = derangement(n)
        ratio = value / math.factorial(n)
        show_math_result(
            "Derangements",
            rf"D_{{{n}}}={value:,}",
            rf"\frac{{D_{{{n}}}}}{{{n}!}}={ratio:.6f}",
            r"e^{-1}\approx0.367879\qquad\text{(large-}n\text{ benchmark)}",
        )


def update_onto(*_):
    with onto_output:
        clear_output(wait=True)
        n, m = onto_n.value, onto_m.value
        value = onto_count(n, m)
        show_math_result(
            "Onto functions",
            rf"\sum_{{j=0}}^{{{m}}}(-1)^j\binom{{{m}}}{{j}}({m}-j)^{{{n}}}={value:,}",
            note="The value is zero when the domain is too small to hit every target.",
        )


der_n.observe(update_derangement, names="value")
onto_n.observe(update_onto, names="value")
onto_m.observe(update_onto, names="value")

display(widgets.HBox([
    widgets.VBox([der_n, der_output], layout=widgets.Layout(width="48%")),
    widgets.VBox([onto_n, onto_m, onto_output], layout=widgets.Layout(width="48%")),
]))
update_derangement()
update_onto()

## 8. Finite uniform probability and hypergeometric sampling

For a uniform finite sample space $$\Omega$$ and an event $$A$$,

$$
\mathbb P(A)=\frac{|A|}{|\Omega|}.
$$

If a population of $$N$$ distinct objects contains $$K$$ marked objects and $$n$$ objects are sampled without replacement, then

$$
\mathbb P(X=k)=
\frac{\binom Kk\binom{N-K}{n-k}}{\binom Nn}.
$$

The value of $$k$$ must belong to the feasible support. Both calculators display exact combinatorial expressions and numerical probabilities.

In [ ]:
uniform_total = widgets.IntSlider(value=20, min=1, max=200, description="|Ω|")
uniform_event = widgets.IntSlider(value=7, min=0, max=200, description="|A|")
uniform_output = widgets.Output()

hyper_N = widgets.IntSlider(value=100, min=1, max=500, description="N")
hyper_K = widgets.IntSlider(value=20, min=0, max=200, description="K")
hyper_n = widgets.IntSlider(value=10, min=0, max=100, description="n")
hyper_k = widgets.IntSlider(value=2, min=0, max=100, description="k")
hyper_output = widgets.Output()


def update_uniform(*_):
    with uniform_output:
        clear_output(wait=True)
        total, event = uniform_total.value, uniform_event.value
        if event > total:
            display(HTML("<b style='color:#a00'>An event cannot contain more outcomes than Ω.</b>"))
            return
        show_math_result(
            "Uniform finite probability",
            rf"\mathbb P(A)=\frac{{|A|}}{{|\Omega|}}=\frac{{{event}}}{{{total}}}={event/total:.6f}",
        )


def update_hypergeometric(*_):
    with hyper_output:
        clear_output(wait=True)
        N, K, n, k = hyper_N.value, hyper_K.value, hyper_n.value, hyper_k.value
        if K > N or n > N:
            display(HTML("<b style='color:#a00'>Require K ≤ N and n ≤ N.</b>"))
            return
        lower, upper = max(0, n - (N - K)), min(n, K)
        if not (lower <= k <= upper):
            show_math_result(
                "Hypergeometric probability",
                rf"k={k}\notin\{{{lower},\ldots,{upper}\}}\quad\Longrightarrow\quad\mathbb P(X=k)=0",
            )
            return
        numerator = math.comb(K, k) * math.comb(N - K, n - k)
        denominator = math.comb(N, n)
        probability = numerator / denominator
        show_math_result(
            "Hypergeometric probability",
            rf"\mathbb P(X={k})="
            rf"\frac{{\binom{{{K}}}{{{k}}}\binom{{{N-K}}}{{{n-k}}}}}"
            rf"{{\binom{{{N}}}{{{n}}}}}={probability:.8f}",
            rf"\text{{support: }}\quad {lower}\le k\le {upper}",
        )


for control in (uniform_total, uniform_event):
    control.observe(update_uniform, names="value")
for control in (hyper_N, hyper_K, hyper_n, hyper_k):
    control.observe(update_hypergeometric, names="value")

display(widgets.VBox([
    widgets.HTML("<h4>Uniform finite model</h4>"),
    widgets.HBox([uniform_total, uniform_event]),
    uniform_output,
    widgets.HTML("<h4>Hypergeometric model</h4>"),
    widgets.HBox([hyper_N, hyper_K, hyper_n, hyper_k]),
    hyper_output,
]))
update_uniform()
update_hypergeometric()

## 9. Birthday collisions

Suppose $$k$$ labelled individuals independently receive one of $$n$$ equally likely dates. There are $$n^k$$ ordered date sequences. When $$k\le n$$, exactly $$(n)_k$$ sequences have no repeated date. Therefore

$$
\mathbb P(\text{at least one collision})
=1-\frac{(n)_k}{n^k}.
$$

For $$k>n$$, the pigeonhole principle makes a collision certain. Move the sliders and observe the threshold in the plot.

In [ ]:
birthday_n = widgets.IntSlider(value=365, min=2, max=500, description="dates n")
birthday_k = widgets.IntSlider(value=23, min=0, max=100, description="people k")
birthday_output = widgets.Output()


def update_birthday(*_):
    with birthday_output:
        clear_output(wait=True)
        n, k = birthday_n.value, birthday_k.value
        probability = birthday_collision_probability(k, n)
        if k <= n:
            display(Math(
                rf"\mathbb P(\text{{collision}})=1-\frac{{({n})_{{{k}}}}}{{{n}^{{{k}}}}}"
                rf"={probability:.6f}"
            ))
        else:
            display(Math(rf"k={k}>n={n}\quad\Longrightarrow\quad\mathbb P(\text{{collision}})=1"))

        maximum = min(100, n + 1)
        xs = np.arange(0, maximum + 1)
        ys = [birthday_collision_probability(int(x), n) for x in xs]
        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.plot(xs, ys, color="black", linewidth=2)
        ax.axhline(0.5, color="gray", linestyle="--", label="probability 0.5")
        ax.axvline(min(k, maximum), color="black", linestyle=":", label=f"k={k}")
        ax.set_ylim(-0.02, 1.02)
        ax.set_xlabel("number of individuals k")
        ax.set_ylabel("collision probability")
        ax.set_title(f"Birthday-collision probability with n={n} dates")
        ax.legend()
        plt.show()


for control in (birthday_n, birthday_k):
    control.observe(update_birthday, names="value")

display(widgets.VBox([
    widgets.HBox([birthday_n, birthday_k]),
    birthday_output,
]))
update_birthday()

## 10. Guided exercise generator

This section creates a new numerical exercise whenever you press **New exercise**. Work on paper first. You may request a hint, enter an integer answer, or reveal a full MathJax solution.

The generator uses four Chapter 1 structures:

$$
\text{complement counting},\qquad
\text{committees},\qquad
\text{stars and bars},\qquad
\text{sampling schemes}.
$$

In [ ]:
exercise_rng = random.Random(20260809)
exercise_type = widgets.Dropdown(
    options=[
        ("Random type", "random"),
        ("Complement counting", "complement"),
        ("Committee", "committee"),
        ("Stars and bars", "stars"),
        ("Sampling scheme", "sampling"),
    ],
    value="random",
    description="Type",
)
new_exercise_button = widgets.Button(description="New exercise", button_style="primary")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal solution")
check_button = widgets.Button(description="Check answer", button_style="success")
exercise_answer = widgets.Text(description="Answer")
exercise_prompt_output = widgets.Output()
exercise_feedback_output = widgets.Output()
exercise_state = {}


def make_exercise(_=None):
    chosen = exercise_type.value
    if chosen == "random":
        chosen = exercise_rng.choice(["complement", "committee", "stars", "sampling"])

    if chosen == "complement":
        q = exercise_rng.randint(5, 10)
        length = exercise_rng.randint(4, 7)
        answer = q ** length - (q - 1) ** length
        prompt = (
            f"An alphabet has **{q}** symbols. How many strings of length **{length}** "
            "contain at least one occurrence of one prescribed symbol?"
        )
        hint = "Count all strings and subtract those that avoid the prescribed symbol."
        solution = rf"{q}^{{{length}}}-({q}-1)^{{{length}}}={answer:,}"

    elif chosen == "committee":
        a = exercise_rng.randint(6, 12)
        b = exercise_rng.randint(5, 10)
        size = exercise_rng.randint(4, min(7, a + b))
        from_a = exercise_rng.randint(max(0, size - b), min(size, a))
        answer = math.comb(a, from_a) * math.comb(b, size - from_a)
        prompt = (
            f"A group contains **{a}** type-A and **{b}** type-B members. "
            f"How many committees of size **{size}** contain exactly **{from_a}** type-A members?"
        )
        hint = "Choose the required members independently from the two disjoint groups."
        solution = (
            rf"\binom{{{a}}}{{{from_a}}}\binom{{{b}}}{{{size-from_a}}}={answer:,}"
        )

    elif chosen == "stars":
        total = exercise_rng.randint(5, 15)
        parts = exercise_rng.randint(2, 5)
        answer = math.comb(total + parts - 1, parts - 1)
        prompt = (
            f"How many non-negative integer solutions does "
            rf"$$x_1+\cdots+x_{{{parts}}}={total}$$ have?"
        )
        hint = "Use stars and bars with r stars and m−1 bars."
        solution = (
            rf"\binom{{{total}+{parts}-1}}{{{parts}-1}}="
            rf"\binom{{{total+parts-1}}}{{{parts-1}}}={answer:,}"
        )

    else:
        n = exercise_rng.randint(5, 12)
        k = exercise_rng.randint(2, min(5, n))
        ordered = exercise_rng.choice([True, False])
        replace = exercise_rng.choice([True, False])
        if ordered and replace:
            answer = n ** k
            formula = rf"{n}^{{{k}}}={answer:,}"
        elif ordered:
            answer = falling_factorial(n, k)
            formula = rf"({n})_{{{k}}}={answer:,}"
        elif replace:
            answer = math.comb(n + k - 1, k)
            formula = rf"\binom{{{n+k-1}}}{{{k}}}={answer:,}"
        else:
            answer = math.comb(n, k)
            formula = rf"\binom{{{n}}}{{{k}}}={answer:,}"
        prompt = (
            f"There are **{n}** available objects or types and the sample size is **{k}**. "
            f"The sample is **{'ordered' if ordered else 'unordered'}** and "
            f"**{'with' if replace else 'without'} replacement**. How many samples are possible?"
        )
        hint = "First locate the case in the four-scheme sampling table."
        solution = formula

    exercise_state.update(answer=answer, hint=hint, solution=solution)
    exercise_answer.value = ""
    with exercise_prompt_output:
        clear_output(wait=True)
        display(Markdown("### Exercise\n\n" + prompt))
    with exercise_feedback_output:
        clear_output()


def check_exercise(_):
    with exercise_feedback_output:
        clear_output(wait=True)
        try:
            submitted = int(exercise_answer.value.replace(",", "").strip())
        except ValueError:
            display(HTML("<b style='color:#a00'>Enter an integer answer.</b>"))
            return
        if submitted == exercise_state["answer"]:
            display(HTML("<b style='color:#075'>Correct. Well done.</b>"))
        else:
            display(HTML("<b style='color:#a00'>Not yet. Recheck the sample structure.</b>"))


def show_hint(_):
    with exercise_feedback_output:
        clear_output(wait=True)
        display(Markdown("**Hint.** " + exercise_state["hint"]))


def reveal_solution(_):
    with exercise_feedback_output:
        clear_output(wait=True)
        display(Markdown("**Solution.**"))
        display(Math(exercise_state["solution"]))


new_exercise_button.on_click(make_exercise)
check_button.on_click(check_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal_solution)

display(widgets.VBox([
    widgets.HBox([exercise_type, new_exercise_button]),
    exercise_prompt_output,
    widgets.HBox([exercise_answer, check_button, hint_button, reveal_button]),
    exercise_feedback_output,
]))
make_exercise()

## 11. Self-check quiz

Choose one answer for each question and press **Grade quiz**. The questions test structural decisions, not only arithmetic. A perfect score requires recognizing the correct sample space before selecting a formula.

Useful formulas include

$$
n^k,\qquad (n)_k,\qquad \binom nk,\qquad
\binom{n+k-1}{k},\qquad
\binom{r+m-1}{m-1}.
$$

In [ ]:
quiz_data = [
    (
        "1. Ordered samples of length 3 from 8 types, with replacement:",
        ["Choose...", "56", "336", "512", "720"],
        "512",
        r"8^3=512",
    ),
    (
        "2. Four-element subsets of a ten-element set:",
        ["Choose...", "40", "210", "5040", "10000"],
        "210",
        r"\binom{10}{4}=210",
    ),
    (
        "3. Non-negative solutions of x₁+x₂+x₃=7:",
        ["Choose...", "21", "28", "36", "45"],
        "36",
        r"\binom{7+3-1}{3-1}=\binom92=36",
    ),
    (
        "4. Derangements of four distinct objects:",
        ["Choose...", "6", "8", "9", "12"],
        "9",
        r"D_4=9",
    ),
    (
        "5. If k>n in the birthday model, the collision probability is:",
        ["Choose...", "0", "1/2", "(n)_k/n^k", "1"],
        "1",
        r"k>n\Longrightarrow\mathbb P(\mathrm{collision})=1",
    ),
]

quiz_widgets = []
quiz_rows = []
for prompt, options, _, _ in quiz_data:
    dropdown = widgets.Dropdown(options=options, value="Choose...", layout=widgets.Layout(width="180px"))
    quiz_widgets.append(dropdown)
    quiz_rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:620px'>{prompt}</div>"),
        dropdown,
    ]))

grade_button = widgets.Button(description="Grade quiz", button_style="primary")
quiz_output = widgets.Output()


def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)
        score = sum(widget.value == data[2] for widget, data in zip(quiz_widgets, quiz_data))
        color = "#075" if score == len(quiz_data) else "#a55"
        display(HTML(
            f"<div style='border:2px solid {color};padding:10px'>"
            f"<b>Score: {score}/{len(quiz_data)}</b></div>"
        ))
        for index, (widget, data) in enumerate(zip(quiz_widgets, quiz_data), start=1):
            symbol = "✓" if widget.value == data[2] else "✗"
            display(Markdown(f"**{symbol} Question {index}**"))
            display(Math(data[3]))


grade_button.on_click(grade_quiz)
display(widgets.VBox(quiz_rows + [grade_button, quiz_output]))

## 12. Automatic mathematical verification

The final code cell performs exact checks of several identities used by the interactive tools. These tests do not replace proofs, but they detect implementation mistakes. Among the checked statements are Vandermonde's identity,

$$
\sum_{k=0}^{n}\binom rk\binom s{n-k}=\binom{r+s}{n},
$$

normalization of a hypergeometric distribution, and benchmark values for derangements and sampling counts.

In [ ]:
# Sampling benchmarks
assert 5 ** 3 == 125
assert falling_factorial(5, 3) == 60
assert math.comb(5 + 3 - 1, 3) == 35
assert math.comb(5, 3) == 10

# Pascal and Vandermonde identities
for n in range(1, 12):
    for k in range(1, n + 1):
        assert math.comb(n + 1, k) == math.comb(n, k) + math.comb(n, k - 1)

for r in range(1, 8):
    for s in range(1, 8):
        for n in range(r + s + 1):
            lhs = sum(
                math.comb(r, k) * math.comb(s, n - k)
                for k in range(n + 1)
                if k <= r and n - k <= s
            )
            assert lhs == math.comb(r + s, n)

# Derangement benchmarks
assert [derangement(n) for n in range(7)] == [1, 0, 1, 2, 9, 44, 265]

# Hypergeometric normalization
N, K, sample_n = 30, 9, 7
support = range(max(0, sample_n - (N - K)), min(sample_n, K) + 1)
probability_sum = sum(
    math.comb(K, k) * math.comb(N - K, sample_n - k) / math.comb(N, sample_n)
    for k in support
)
assert abs(probability_sum - 1.0) < 1e-12

show_math_result(
    "All automatic checks passed",
    r"\sum_k \mathbb P(X=k)=1",
    r"\text{Pascal, Vandermonde, sampling and derangement checks: valid}",
    note="The notebook is ready for instructional use in Google Colab.",
)

## Further work

To learn effectively, do not use the notebook only as a calculator. For every experiment:

1. describe the elementary outcomes;
2. decide whether order matters;
3. decide whether repetition is allowed;
4. predict the formula before moving a slider;
5. explain why the computed number counts each outcome exactly once.

The notebook is self-contained and may be placed directly in a GitHub repository and opened with Google Colab.